# Focus Fatigue — Full Pipeline

Thin runner that calls `src.pipeline` — no inline pipeline logic.

## Checkpoint / Resume

Phase 1 saves a JSON checkpoint (`model1_results_checkpoint.json`) after completion.
If the kernel crashes during Phase 2 or 3:
1. Restart the kernel
2. Run cells 1-3 (setup, config, discover)
3. Set `RESUME = True` in cell 1 (`RESUME` variable)
4. Skip Phase 1 cell (or run it — it's idempotent)
5. Run Phase 2 — it will skip matches with existing signal outputs

## Setup
1. Set `DATA_ROOT` to your data directory
2. Optionally set `MATCH_IDS` to limit which matches to process
3. Run all cells


In [ ]:
import sys
import json
from pathlib import Path

# ── POINT THIS TO YOUR DATA DIRECTORY ──
DATA_ROOT = "/mnt/usb/conor_downloads/team_mappings"

# Leave empty to auto-discover, or specify exact match IDs
MATCH_IDS = []  # e.g. ["2215790"] or [] for all

# Optional: limit frames per match for quick testing
TEST_FRAMES = None  # e.g. 5000

# ── Resume mode ──
# After Phase 1 completes and the kernel restarts, set this to True
# to load the checkpoint and skip already-computed signals.
RESUME = False

# ── Resolved paths ──
TRACKING_DIR = Path(DATA_ROOT) / "tracking"
SAMPLE_DIR = Path(DATA_ROOT) / "sample"

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Data root:    {DATA_ROOT}")
print(f"Tracking:     {TRACKING_DIR}")
print(f"Sample:       {SAMPLE_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Resume mode:  {RESUME}")


In [ ]:
# ── Configure paths ──
from src.pressure.config import DEFAULT_CONFIG
from src.signals.config import DEFAULT_SIGNAL_CONFIG

# Override paths in config
DEFAULT_CONFIG.tracking_dir = str(TRACKING_DIR)
DEFAULT_CONFIG.sample_dir = str(SAMPLE_DIR)
DEFAULT_CONFIG.output_dir = str(PROJECT_ROOT / "outputs" / "pressure_exposure")
DEFAULT_SIGNAL_CONFIG.output_root = str(PROJECT_ROOT / "outputs" / "signals")

# Also set team mappings path used by drift signal
import src.signals.drift as drift_module
team_map_path = Path(DATA_ROOT) / "team_mappings" / "team_mappings.csv"
if not team_map_path.exists():
    team_map_path = Path(DATA_ROOT) / "team_mappings.csv"
drift_module._TEAM_MAPPINGS_PATH = team_map_path

import os
os.makedirs(DEFAULT_CONFIG.output_dir, exist_ok=True)
os.makedirs(DEFAULT_SIGNAL_CONFIG.output_root, exist_ok=True)
for sig in ["positional_drift", "shift_latency", "pressing_accuracy", "transition_latency", "physical_load"]:
    os.makedirs(os.path.join(DEFAULT_SIGNAL_CONFIG.output_root, sig), exist_ok=True)

print(f"Pressure output: {DEFAULT_CONFIG.output_dir}")
print(f"Signals output:  {DEFAULT_SIGNAL_CONFIG.output_root}")
print(f"Team map:        {team_map_path} (exists: {team_map_path.exists()})")

In [ ]:
# ── Discover matches ──
if MATCH_IDS:
    matches = MATCH_IDS
else:
    matches = sorted([d.name for d in TRACKING_DIR.iterdir()
                      if d.is_dir() and (d / "tracking.parquet").exists()])
    if not matches:
        matches = sorted([d.name for d in SAMPLE_DIR.iterdir()
                          if d.is_dir() and (d / "tracking.parquet").exists()])

if not matches:
    print("❌ No matches found!")
else:
    print(f"Found {len(matches)} matches to process:")
    for m in matches[:5]:
        print(f"  • {m}")
    if len(matches) > 5:
        print(f"  ... and {len(matches)-5} more")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 1: Model 1 — Pressure Exposure
# ═══════════════════════════════════════════════════════════════

from src.pipeline import run_model1_on_match

print("=" * 70)
print("PHASE 1: MODEL 1 — PRESSURE EXPOSURE")
print("=" * 70)

model1_results = {}
for idx, match_id in enumerate(matches):
    match_path = TRACKING_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        match_path = SAMPLE_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        print(f"  ⚠️  [{idx+1}/{len(matches)}] {match_id}: not found, skipping")
        continue
    result = run_model1_on_match(match_id, match_path, DEFAULT_CONFIG, nrows=TEST_FRAMES)
    model1_results[match_id] = result

print(f"\n{'=' * 70}")
print(f"Model 1 complete: {len(model1_results)} matches")
print(f"{'=' * 70}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 1: Model 1 — Pressure Exposure
# ═══════════════════════════════════════════════════════════════
# If RESUME=True, load checkpoint instead of re-running.
# If RESUME=False, run Model 1 and save checkpoint.

CHECKPOINT_PATH = PROJECT_ROOT / "outputs" / "model1_results_checkpoint.json"

if RESUME and CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH) as f:
        model1_results_list = json.load(f)
    model1_results = {r["match_id"]: r for r in model1_results_list}
    print("=" * 70)
    print("PHASE 1: CHECKPOINT LOADED (resume mode)")
    print("=" * 70)
    print(f"  Loaded {len(model1_results)} matches from checkpoint")
else:
    from src.pipeline import run_model1_on_match

    print("=" * 70)
    print("PHASE 1: MODEL 1 — PRESSURE EXPOSURE")
    print("=" * 70)

    model1_results = {}
    for idx, match_id in enumerate(matches):
        match_path = TRACKING_DIR / match_id / "tracking.parquet"
        if not match_path.exists():
            match_path = SAMPLE_DIR / match_id / "tracking.parquet"
        if not match_path.exists():
            print(f"  ⚠️  [{idx+1}/{len(matches)}] {match_id}: not found, skipping")
            continue
        result = run_model1_on_match(match_id, match_path, DEFAULT_CONFIG, nrows=TEST_FRAMES)
        model1_results[match_id] = result

    # ── Save checkpoint ──
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    serialisable = []
    for mid, r in model1_results.items():
        serialisable.append({
            "match_id": mid,
            "n_players": r.get("n_players", 0),
            "n_blocks": r.get("n_blocks", 0),
            "high": r.get("high", 0),
            "low": r.get("low", 0),
            "elapsed_s": r.get("elapsed_s", 0),
        })
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(serialisable, f, indent=2)
    print(f"  ✅ Checkpoint saved: {CHECKPOINT_PATH}")
    print(f"     {len(serialisable)} matches, {sum(r.get('n_blocks', 0) for r in serialisable)} blocks")

print(f"\n{'=' * 70}")
print(f"Model 1: {len(model1_results)} matches")
print(f"{'=' * 70}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 2: All Signals
# ═══════════════════════════════════════════════════════════════
# If RESUME=True, skip matches that already have all signal outputs.

from src.pipeline import run_signals_on_match, list_signal_descriptions
from src.signals.registry import list_signals

print("=" * 70)
print("PHASE 2: ALL SIGNALS")
print("=" * 70)

registered = list_signals()
print(f"Registered signals ({len(registered)}): {registered}\n")

signal_results = {}

for idx, match_id in enumerate(matches):
    match_path = TRACKING_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        match_path = SAMPLE_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        continue

    # ── Resume check: skip if all signal CSVs already exist ──
    if RESUME:
        all_exist = True
        for sig in registered:
            sig_path = Path(DEFAULT_SIGNAL_CONFIG.output_root) / sig / f"{match_id}.csv"
            if not sig_path.exists():
                all_exist = False
                break
        if all_exist:
            print(f"  ⏭️  [{idx+1}/{len(matches)}] {match_id}: all signals exist, skipping")
            continue
        print(f"  ▶️  [{idx+1}/{len(matches)}] {match_id}: computing missing signals")

    result = run_signals_on_match(match_id, match_path, DEFAULT_CONFIG, DEFAULT_SIGNAL_CONFIG,
                                   str(TRACKING_DIR), nrows=TEST_FRAMES)
    signal_results[match_id] = result

print(f"\n{'=' * 70}")
print(f"Signals complete: {len(signal_results)} matches")
print(f"{'=' * 70}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3: Merge — Unified Fatigue Dataset
# ══════════════════════════════════════════════════════════════════════════════
# Merges Model 1 pressure outputs with all signal CSVs into a single parquet.
# No resume logic needed — merge is idempotent (overwrites output file).

from src.merge_outputs import merge_all

print("=" * 70)
print("PHASE 3: MERGE — UNIFIED FATIGUE DATASET")
print("=" * 70)

unified_path = str(PROJECT_ROOT / "outputs" / "unified_fatigue_dataset.parquet")

merged_df = merge_all(
    signals_dir=DEFAULT_SIGNAL_CONFIG.output_root,
    pressure_dir=DEFAULT_CONFIG.output_dir,
    output_path=unified_path,
)

rows = len(merged_df) if hasattr(merged_df, "__len__") else 0
print(f"\n{'=' * 70}")
print(f"Phase 3 complete — merged {rows} rows" if rows > 0 else "Phase 3 complete — no data to merge")
print(f"{'=' * 70}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════

print("=" * 70)
print("PIPELINE COMPLETE — SUMMARY")
print("=" * 70)
print(f"  Matches attempted: {len(matches)}")
print(f"  Model 1 success:   {len(model1_results)}")
print(f"  Signals success:   {len(signal_results)}")
print()

if signal_results:
    print(f"  {'Signal':<25} {'Rows':>8} {'Errors':>8}")
    print(f"  {'─' * 41}")
    for sig_name in registered:
        total_rows = sum(r["signals"].get(sig_name, {}).get("rows", 0) for r in signal_results.values())
        errors = [mid for mid, r in signal_results.items() if r["signals"].get(sig_name, {}).get("error")]
        print(f"  {sig_name:<25} {total_rows:>8,} {len(errors) if errors else '─':>8}")

print()
print(f"  Output files:")
print(f"    Pressure:  {DEFAULT_CONFIG.output_dir}/")
print(f"    Signals:   {DEFAULT_SIGNAL_CONFIG.output_root}/{{signal}}/{{match}}.csv")
print(f"    Unified:   {PROJECT_ROOT / 'outputs' / 'unified_fatigue_dataset.parquet'}")